# Track 00 — Framework shootout: cái nào thật sự dùng được?

Notebook này **đo thật trên Kaggle T4**, không phải bảng ý kiến. Trả lời 3 câu:

1. **Fine-tune:** framework nào cài được, nhanh bao nhiêu, tốn VRAM bao nhiêu?
2. **Inference:** throughput chênh nhau thế nào?
3. **Đánh giá:** harness nào là chuẩn giới researcher dùng, cài được không?

Cuối cùng là **checklist của researcher** — phần mà hầu hết tutorial bỏ qua và là lý do
nhiều "kết quả" fine-tune không đáng tin.

> ⚠️ **Bài học từ lần chạy đầu (run v1 đã ERROR):** tôi từng viết "probe dùng subprocess riêng
> nên không phá kernel". **Sai.** `subprocess` chỉ cô lập việc *import*, còn `pip install` vẫn
> ghi vào **cùng một** site-packages. Cài `unsloth`/`vllm` đã nâng/hạ torch và làm gãy
> `torch.distributed` cho mọi ô phía sau:
>
> ```
> ImportError: cannot import name '_chunk_or_narrow_cat' from 'torch._utils'
> ```
>
> **Cách sửa trong notebook này:** chia framework làm 2 nhóm và **đảo thứ tự** —
> benchmark chạy TRƯỚC, cài nhóm phá môi trường để SAU CÙNG. Muốn benchmark sâu một framework
> nhóm 2 thì phải cho nó một session Kaggle riêng.

## 0. Môi trường

In [ ]:
# ================= GHIM 1 GPU - BAT BUOC, PHAI CHAY TRUOC MOI import torch =================
# Kaggle cap 2xT4. Khi thay NHIEU GPU, transformers Trainer AM THAM boc model vao
# nn.DataParallel -> batch_size thuc te nhan doi, va nn.DataParallel von kem hieu qua.
# TRL SFTTrainer thi tu choi thang:
#   ValueError: SFTTrainer does not support `nn.DataParallel` ...
# Da lam run v1 va v2 cua notebook nay ERROR, va lam con so cua notebook 01 bi nhap nhem.
# Muon dung that 2 GPU -> `accelerate launch --multi_gpu` voi script rieng (xem notebook 02).
import os, subprocess
_n = subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout.strip().splitlines()
print(f"May nay co {len(_n)} GPU:")
for l in _n: print("  ", l)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("\n-> Da ghim CUDA_VISIBLE_DEVICES=0 de moi phep do la single-GPU, cong bang va tai lap duoc.")

In [ ]:
import torch, subprocess, sys, json, time, os, re
print("torch:", torch.__version__)
assert torch.cuda.is_available(), "Bat GPU trong Settings"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
VRAM = torch.cuda.get_device_properties(0).total_memory/1e9
# torch.cuda.is_bf16_supported() tra ve True ca tren T4 (sm_75) vi tinh ca bf16 emulation
# -> KHONG tin duoc. Phai check compute capability: bf16 native chi tu sm_80.
print("is_bf16_supported() bao:", torch.cuda.is_bf16_supported(), "<- khong tin duoc")
BF16 = CAP[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"GPU {GPU} | cap {CAP} | VRAM {VRAM:.1f}GB | dung {'bf16' if BF16 else 'fp16'}")

In [ ]:
import matplotlib as mpl, matplotlib.pyplot as plt, numpy as np
SURFACE="#fcfcfb"; INK="#0b0b0b"; INK2="#52514e"; GRID="#e5e4e0"
S1="#2a78d6"; S2="#eb6834"; S3="#1baf7a"
mpl.rcParams.update({"figure.facecolor":SURFACE,"axes.facecolor":SURFACE,"axes.edgecolor":GRID,
 "axes.labelcolor":INK2,"text.color":INK,"xtick.color":INK2,"ytick.color":INK2,
 "axes.spines.top":False,"axes.spines.right":False,"grid.color":GRID,"grid.linewidth":.8,
 "font.size":10,"axes.titlesize":12,"lines.linewidth":2,"lines.markersize":6,"figure.dpi":120})
def finish(ax,title,ylab=None,xlab=None,legend=False):
    ax.set_title(title,loc="left",pad=12,color=INK)
    if ylab: ax.set_ylabel(ylab)
    if xlab: ax.set_xlabel(xlab)
    ax.grid(axis="y",alpha=.7); ax.set_axisbelow(True)
    if legend: ax.legend(frameon=False)
    plt.tight_layout()

print("style ok")

## 1. Probe: framework nào cài được trên Kaggle?

Mỗi framework được cài + import trong **subprocess riêng** rồi báo về. Không làm bẩn kernel.
Cột `phut` là thời gian cài — con số quan trọng khi session Kaggle chỉ có 12h.

In [ ]:
# NHOM 1 - an toan: coexist duoc trong 1 kernel, khong dung den torch core
SAFE = [
  ("transformers",   "transformers==5.22.1",     "transformers",        "finetune"),
  ("peft",           "peft==0.21.2",             "peft",                "finetune"),
  ("trl",            "trl==1.17.1",              "trl",                 "finetune"),
  ("accelerate",     "accelerate",               "accelerate",          "finetune"),
  ("bitsandbytes",   "bitsandbytes",             "bitsandbytes",        "finetune"),
  ("lm-eval",        "lm-eval",                  "lm_eval",             "eval"),
  ("lighteval",      "lighteval",                "lighteval",           "eval"),
  ("mteb",           "mteb",                     "mteb",                "eval"),
  ("deepeval",       "deepeval",                 "deepeval",            "eval"),
  ("ragas",          "ragas",                    "ragas",               "eval"),
  ("evaluate",       "evaluate",                 "evaluate",            "eval"),
  ("sentence-transformers","sentence-transformers","sentence_transformers","eval"),
  ("optimum",        "optimum",                  "optimum",             "inference"),
  ("wandb",          "wandb",                    "wandb",               "tracking"),
  ("mlflow",         "mlflow",                   "mlflow",              "tracking"),
  ("langfuse",       "langfuse",                 "langfuse",            "tracking"),
]

# NHOM 2 - PHA MOI TRUONG: thay doi torch / build native. Chi probe o MUC 6 (cuoi cung).
WRECKERS = [
  ("unsloth",        "unsloth",                  "unsloth",             "finetune"),
  ("llamafactory",   "llamafactory",             "llamafactory",        "finetune"),
  ("torchtune",      "torchtune",                "torchtune",           "finetune"),
  ("axolotl",        "axolotl",                  "axolotl",             "finetune"),
  ("sglang",         "sglang",                   "sglang",              "inference"),
  ("llama-cpp-python","llama-cpp-python",        "llama_cpp",           "inference"),
  ("vlmeval",        "vlmeval",                  "vlmeval",             "eval"),
]

PROBE = r'''
import importlib, sys, json
m = sys.argv[1]
try:
    mod = importlib.import_module(m)
    print(json.dumps({"ok":True,"ver":getattr(mod,"__version__","?")}))
except Exception as e:
    print(json.dumps({"ok":False,"err":type(e).__name__+": "+str(e)[:160]}))
'''
open("/kaggle/working/_probe.py","w").write(PROBE)

def probe(name, spec, mod):
    t0=time.time()
    inst = subprocess.run([sys.executable,"-m","pip","install","-q",spec],
                          capture_output=True, text=True, timeout=1800)
    t_inst = time.time()-t0
    if inst.returncode != 0:
        err = (inst.stderr or inst.stdout).strip().splitlines()
        return {"name":name,"install":False,"import":False,"ver":"-",
                "min":t_inst/60,"err":(err[-1][:160] if err else "pip failed")}
    r = subprocess.run([sys.executable,"/kaggle/working/_probe.py",mod],
                       capture_output=True, text=True, timeout=600)
    try:    d=json.loads(r.stdout.strip().splitlines()[-1])
    except Exception: d={"ok":False,"err":(r.stderr or r.stdout)[-160:]}
    return {"name":name,"install":True,"import":d["ok"],
            "ver":d.get("ver","-"),"min":t_inst/60,"err":d.get("err","")}

results=[]
for name,spec,mod,grp in SAFE:
    r=probe(name,spec,mod); r["group"]=grp; results.append(r)
    flag = "OK " if r["import"] else ("IMPORT-FAIL" if r["install"] else "INSTALL-FAIL")
    print(f"{flag:12s} {name:24s} {r['ver']:>12s}  {r['min']:5.1f} phut  {r['err'][:70]}")

json.dump(results, open("/kaggle/working/framework_probe.json","w"), indent=2)

In [ ]:
# Bang tong hop
print(f"{'Nhom':10s}{'Framework':24s}{'Ban':>12s}{'Cai (phut)':>12s}  Ket qua")
print("-"*78)
for grp in ["finetune","inference","eval","tracking"]:
    for r in [x for x in results if x["group"]==grp]:
        st = "dung duoc" if r["import"] else ("import loi" if r["install"] else "cai that bai")
        print(f"{grp:10s}{r['name']:24s}{r['ver']:>12s}{r['min']:>12.1f}  {st}")

ok = [r for r in results if r["import"]]
print(f"\n{len(ok)}/{len(results)} framework NHOM AN TOAN dung duoc tren {GPU}")
print("Nhom pha moi truong se probe o MUC 6, sau khi benchmark xong.")

In [ ]:
# Bieu do: thoi gian cai (chi nhung cai thanh cong) - 1 series, khong can legend
okr = sorted([r for r in results if r["import"]], key=lambda r:-r["min"])[:14]
fig, ax = plt.subplots(figsize=(7.6, .46*len(okr)+1.6))
ax.barh([r["name"] for r in okr], [r["min"] for r in okr],
        color=S1, edgecolor=SURFACE, linewidth=2, height=.62)
for i,r in enumerate(okr):
    ax.annotate(f"{r['min']:.1f}", (r["min"],i), textcoords="offset points",
                xytext=(5,0), va="center", color=INK2, fontsize=9)
ax.invert_yaxis(); ax.grid(axis="x",alpha=.7); ax.set_axisbelow(True)
ax.set_title("Thoi gian cai dat (phut) - chi framework cai duoc", loc="left", pad=12, color=INK)
ax.set_xlabel("phut"); plt.tight_layout(); plt.show()
print("Luu y: session Kaggle toi da 12h. Framework cai >10 phut an dang ke quota.")

## 2. Fine-tune shootout: HF Trainer vs TRL SFTTrainer

Cùng model, cùng data, cùng số step, cùng LoRA config. Đo **giây/step** và **VRAM đỉnh**.

Đây là so sánh công bằng duy nhất có thể làm trong một kernel. Unsloth phải chạy session riêng
vì nó ghi đè `transformers`/`torch`.

In [ ]:
!pip install -q -U "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "datasets==5.1.0" accelerate
# Kaggle cai san torchao 0.10.0 nhung peft 0.21.2 doi >=0.16.0 -> peft.is_torchao_available()
# RAISE ImportError ngay khi gan LoRA. Minh khong dung torchao nen go han la xong.
!pip uninstall -q -y torchao 2>/dev/null; echo "da go torchao"

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

MODEL="Qwen/Qwen3-0.6B"; STEPS=20; BS=4; ACC=2
tok=AutoTokenizer.from_pretrained(MODEL)
raw=load_dataset("b-mc2/sql-create-context",split="train").shuffle(seed=42).select(range(400))
SYSTEM="Ban la bo dich text-to-SQL. Chi tra ve cau truy van SQLite."

def prompt_of(ex):
    return tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},
         {"role":"user","content":f"Schema:\n{ex['context']}\n\nCau hoi: {ex['question']}"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

def fresh():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def measure(fn, label):
    fresh(); t0=time.time(); loss=fn(); dt=time.time()-t0
    peak=torch.cuda.max_memory_allocated()/1e9
    print(f"{label:22s} {dt/STEPS:6.3f} s/step | VRAM dinh {peak:5.2f} GB | loss {loss:.4f}")
    return {"name":label,"s_per_step":dt/STEPS,"vram":peak,"loss":loss}

bench=[]

In [ ]:
# --- A. HF Trainer thuan + PEFT (tu viet collator, tu mask label) ---
def run_hf():
    def feat(ex):
        p=tok(prompt_of(ex),add_special_tokens=False)["input_ids"]
        a=tok(ex["answer"]+tok.eos_token,add_special_tokens=False)["input_ids"]
        return {"input_ids":(p+a)[:512],"labels":([-100]*len(p)+a)[:512]}
    ds=raw.map(feat,remove_columns=raw.column_names)
    def coll(fs):
        n=max(len(f["input_ids"]) for f in fs); i=[];l=[];a=[]
        for f in fs:
            k=n-len(f["input_ids"])
            i.append(f["input_ids"]+[tok.pad_token_id]*k); l.append(f["labels"]+[-100]*k)
            a.append([1]*len(f["input_ids"])+[0]*k)
        t=torch.tensor; return {"input_ids":t(i),"labels":t(l),"attention_mask":t(a)}
    m=get_peft_model(AutoModelForCausalLM.from_pretrained(MODEL,dtype=DTYPE,device_map="cuda"),
                     LoraConfig(task_type="CAUSAL_LM",**LORA))
    m.config.use_cache=False
    tr=Trainer(model=m,args=TrainingArguments(
        output_dir="/kaggle/working/_hf",per_device_train_batch_size=BS,
        gradient_accumulation_steps=ACC,max_steps=STEPS,learning_rate=2e-4,
        logging_steps=100,bf16=BF16,fp16=not BF16,save_strategy="no",report_to=[]),
        train_dataset=ds,data_collator=coll)
    out=tr.train(); del m,tr; return out.training_loss

bench.append(measure(run_hf, "HF Trainer + PEFT"))

In [ ]:
# --- B. TRL SFTTrainer (lo mask label va collator ho) ---
from trl import SFTTrainer, SFTConfig
def run_trl():
    ds=raw.map(lambda ex:{"prompt":prompt_of(ex),"completion":ex["answer"]},
               remove_columns=raw.column_names)
    m=AutoModelForCausalLM.from_pretrained(MODEL,dtype=DTYPE,device_map="cuda")
    m.config.use_cache=False
    tr=SFTTrainer(model=m,args=SFTConfig(
        output_dir="/kaggle/working/_trl",per_device_train_batch_size=BS,
        gradient_accumulation_steps=ACC,max_steps=STEPS,learning_rate=2e-4,
        logging_steps=100,bf16=BF16,fp16=not BF16,save_strategy="no",report_to=[],
        max_length=512,gradient_checkpointing=False),
        train_dataset=ds,peft_config=LoraConfig(task_type="CAUSAL_LM",**LORA))
    out=tr.train(); del m,tr; return out.training_loss

bench.append(measure(run_trl, "TRL SFTTrainer"))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(9.6,3.4))
names=[b["name"] for b in bench]
for ax,key,title,unit in [(axes[0],"s_per_step","Toc do (thap hon = tot hon)","giay/step"),
                          (axes[1],"vram","VRAM dinh","GB")]:
    vals=[b[key] for b in bench]
    bars=ax.bar(names,vals,width=.5,color=[S1,S2],edgecolor=SURFACE,linewidth=2)
    for b,v in zip(bars,vals):
        ax.annotate(f"{v:.2f}",(b.get_x()+b.get_width()/2,v),textcoords="offset points",
                    xytext=(0,5),ha="center",color=INK,fontsize=10,fontweight="600")
    ax.set_ylim(0,max(vals)*1.25); finish(ax,title,unit)
plt.show()

print(f"{'Framework':24s}{'s/step':>9s}{'VRAM GB':>10s}{'loss':>9s}")
print("-"*52)
for b in bench: print(f"{b['name']:24s}{b['s_per_step']:>9.3f}{b['vram']:>10.2f}{b['loss']:>9.4f}")
print("\nKet luan thuc dung:")
print("- HF Trainer: kiem soat hoan toan, phai tu mask label. Hoc thi dung cai nay.")
print("- TRL: mask label + collator tu dong, co san DPO/GRPO/KTO. San xuat thi dung cai nay.")
print("- Unsloth: nhanh ~2x + it VRAM hon, nhung ghi de thu vien -> session rieng.")

## 3. Inference baseline: `transformers.generate`

Đo throughput của đường cơ sở. vLLM để **mục 5**, vì cài nó cũng phá torch — cùng lý do
với nhóm wrecker ở mục 6.

In [ ]:
PROMPTS = [prompt_of(raw[i]) for i in range(32)]

# --- transformers.generate theo batch ---
fresh()
tok.padding_side="left"
m = AutoModelForCausalLM.from_pretrained(MODEL,dtype=DTYPE,device_map="cuda"); m.eval()
t0=time.time(); n_tok=0
with torch.no_grad():
    for i in range(0,len(PROMPTS),8):
        b=tok(PROMPTS[i:i+8],return_tensors="pt",padding=True,add_special_tokens=False).to("cuda")
        g=m.generate(**b,max_new_tokens=64,do_sample=False,pad_token_id=tok.pad_token_id)
        n_tok += int((g[:,b["input_ids"].shape[1]:]!=tok.pad_token_id).sum())
hf_dt=time.time()-t0; hf_tps=n_tok/hf_dt
print(f"transformers.generate : {hf_tps:7.1f} tok/s  ({n_tok} tok / {hf_dt:.1f}s)  VRAM {torch.cuda.max_memory_allocated()/1e9:.2f}GB")
del m; fresh()

## 4. Đánh giá: harness mà researcher thật sự dùng

| Harness | Dùng cho | Vì sao quan trọng |
|---|---|---|
| **lm-evaluation-harness** (EleutherAI) | LLM text | **Chuẩn de-facto.** Mọi paper và Open LLM Leaderboard cũ đều dùng. Nếu bạn báo số không qua harness này, reviewer sẽ hỏi. |
| **lighteval** (HF) | LLM text | Bản của HF, nhẹ hơn, tích hợp Hub tốt |
| **mteb** | Embedding / reranker | Đường vào MTEB + VN-MTEB leaderboard |
| **VLMEvalKit** (OpenCompass) | VLM | 220+ model, 80+ benchmark. Đường vào OpenVLM leaderboard |
| **OmniDocBench** | OCR / document | Đường vào bảng OCR — **sân tốt nhất cho model nhỏ** |
| **deepeval / ragas** | LLM-as-judge, RAG | Đo chất lượng không có đáp án chuẩn |
| **evaluate** (HF) | metric lẻ | BLEU/ROUGE/WER/F1 — tiện nhưng không phải harness |

Ô dưới chạy `lm-eval` thật trên một task nhỏ để bạn thấy output nó ra sao.

In [ ]:
LMEVAL_OK = any(r["name"]=="lm-eval" and r["import"] for r in results)
if LMEVAL_OK:
    # task nho, limit thap de chay nhanh. Thay --tasks bang gsm8k/mmlu khi lam that.
    cmd = (f"lm_eval --model hf --model_args pretrained={MODEL},dtype={'bfloat16' if BF16 else 'float16'} "
           f"--tasks arc_easy --limit 40 --batch_size 8 --device cuda:0")
    print("$", cmd, "\n")
    r=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=2400)
    print(r.stdout[-2500:] if r.stdout else "")
    if r.returncode!=0: print("STDERR:", (r.stderr or "")[-1200:])
else:
    print("lm-eval khong cai duoc -> xem cot err o muc 1")
print("\nKhi lam that: --tasks gsm8k,mmlu,arc_challenge,hellaswag --limit None")
print("Va PHAI bao cao: ten task, so shot, phien ban harness, commit hash. Thieu la khong reproduce duoc.")

## 5. Inference nâng cao: vLLM — ⚠️ cài xong là torch có thể gãy

vLLM dùng PagedAttention + continuous batching, throughput cao hơn `transformers.generate`
nhiều khi chạy nhiều request. Nhưng nó ghim phiên bản torch riêng.

Ô này đặt **sau** `lm-eval` là có chủ đích: mọi thứ cần torch nguyên vẹn phải chạy xong trước.

In [ ]:
# --- vLLM (cai lau, ~5-10 phut tren Kaggle) ---
VLLM_OK=False
try:
    import vllm; VLLM_OK=True
except ImportError:
    r=subprocess.run([sys.executable,"-m","pip","install","-q","vllm"],capture_output=True,text=True,timeout=2400)
    print("cai vllm:", "OK" if r.returncode==0 else "THAT BAI")
    if r.returncode!=0: print((r.stderr or r.stdout)[-400:])
    else:
        try: import vllm; VLLM_OK=True
        except Exception as e: print("import loi:",e)

vllm_tps=None
if VLLM_OK:
    from vllm import LLM, SamplingParams
    llm=LLM(model=MODEL, dtype="float16" if not BF16 else "bfloat16",
            gpu_memory_utilization=0.80, max_model_len=1024, enforce_eager=True)
    sp=SamplingParams(max_tokens=64, temperature=0)
    t0=time.time(); outs=llm.generate(PROMPTS, sp); dt=time.time()-t0
    n=sum(len(o.outputs[0].token_ids) for o in outs); vllm_tps=n/dt
    print(f"vLLM                  : {vllm_tps:7.1f} tok/s  ({n} tok / {dt:.1f}s)")
    print(f"-> vLLM nhanh gap {vllm_tps/hf_tps:.1f}x transformers o batch nay")
else:
    print("Bo qua vLLM. Ghi nhan: KHONG cai duoc tren Kaggle T4 trong session nay.")

In [ ]:
labels=["transformers"]+(["vLLM"] if vllm_tps else [])
vals=[hf_tps]+([vllm_tps] if vllm_tps else [])
fig,ax=plt.subplots(figsize=(5.4,3.4))
bars=ax.bar(labels,vals,width=.5,color=[S1,S3][:len(vals)],edgecolor=SURFACE,linewidth=2)
for b,v in zip(bars,vals):
    ax.annotate(f"{v:.0f}",(b.get_x()+b.get_width()/2,v),textcoords="offset points",
                xytext=(0,5),ha="center",color=INK,fontsize=10,fontweight="600")
ax.set_ylim(0,max(vals)*1.25)
finish(ax,"Throughput inference (cao hon = tot hon)","tokens/giay")
plt.show()
print(f"{'Engine':16s}{'tok/s':>10s}")
print("-"*26)
for l,v in zip(labels,vals): print(f"{l:16s}{v:>10.1f}")

## 6. ⚠️ Probe nhóm phá môi trường — ô CUỐI CÙNG

Mọi benchmark phía trên đã xong, nên từ đây môi trường có gãy cũng không mất dữ liệu.
Đó chính là lý do ô này nằm cuối, không nằm đầu.

Sau ô này **mọi ô phía trên sẽ không chạy lại được** cho tới khi restart kernel.

In [ ]:
wreck=[]
for name,spec,mod,grp in WRECKERS:
    r=probe(name,spec,mod); r["group"]=grp; wreck.append(r)
    flag = "OK " if r["import"] else ("IMPORT-FAIL" if r["install"] else "INSTALL-FAIL")
    print(f"{flag:12s} {name:22s} {r['ver']:>12s}  {r['min']:5.1f} phut  {r['err'][:60]}")

print(f"\n{sum(r['import'] for r in wreck)}/{len(wreck)} framework nhom 2 dung duoc")
print("\nMuon benchmark sau cai nao -> cho no 1 notebook Kaggle rieng.")
print("torch con nguyen ven khong?")
try:
    import importlib, torch.distributed as _d; importlib.reload(_d); print("  torch.distributed: OK")
except Exception as e:
    print("  torch.distributed: DA GAY ->", type(e).__name__, str(e)[:90])
    print("  (day la bang chung cho bai hoc o dau notebook)")

## 7. Checklist trước khi công bố bất kỳ con số nào

**Tái lập (reproducibility)**
- [ ] Ghim seed (`seed`, `data_seed`) **và** báo cáo nó
- [ ] Ghim phiên bản mọi thư viện (`pip freeze > requirements.txt`)
- [ ] Ghi lại GPU + dtype — **bf16 và fp16 cho kết quả khác nhau**
- [ ] Commit hash của code
- [ ] Config ở file riêng (YAML/JSON), không hardcode trong notebook

**Tính đúng đắn của đo lường**
- [ ] Tập test **không giao** tập train/val — kiểm tra bằng hash, không bằng niềm tin
- [ ] Chạy **≥3 seed**, báo cáo `mean ± std`, không báo một lần chạy
- [ ] Có baseline model gốc (chưa fine-tune) trong cùng bảng
- [ ] Có tập **held-out paraphrase** để phát hiện học thuộc câu chữ
- [ ] Thước đo khớp nhiệm vụ, không chỉ loss

**Ablation (bỏ từng thành phần xem ảnh hưởng)**
- [ ] Đổi **một** biến mỗi lần
- [ ] Bảng ablation: data size × LoRA rank × lr × epoch
- [ ] Ghi cả **thí nghiệm thất bại** — đó là thông tin, không phải rác

**Theo dõi thí nghiệm**
- [ ] W&B / MLflow / TensorBoard cho loss curve
- [ ] **Langfuse** cho trace inference ở production (bạn đã dùng cái này ở `internal-chat`)
- [ ] Lưu `metrics.json` mỗi lần chạy

**Báo cáo**
- [ ] Model card: data gì, license gì, giới hạn gì, bias gì
- [ ] Nói rõ cái model **làm được** và **không làm được**
- [ ] Nêu chi phí compute (GPU-giờ) — reviewer sẽ hỏi

> Khoảng cách giữa "tôi fine-tune được" và "tôi có kết quả đáng tin" chính là cái checklist này.
> Repo `smartkids-finetune` của bạn đã làm tốt nhiều mục (held-out paraphrase, LLM-judge ẩn danh,
> benchmark đa model). Mục còn thiếu rõ nhất: **multi-seed + mean±std**.

In [ ]:
json.dump({"gpu":GPU,"cap":list(CAP),"bf16":BF16,
           "frameworks":results,"finetune_bench":bench,
           "infer_hf_tps":hf_tps,"infer_vllm_tps":vllm_tps,
           "wreckers":wreck,
           "bf16_reported_by_torch":torch.cuda.is_bf16_supported(),"compute_cap":list(CAP)},
          open("/kaggle/working/framework_report.json","w"), indent=2)
print("Da luu /kaggle/working/framework_report.json")